# Single-Pass vs Autoregressive Rollout Schematic (real-data version)

Builds `fig:single_step_vs_autoregressive`, replacing the earlier flat-oval placeholders with
real rendered electric potential maps (same rendering pipeline as the training-strategy figure),
using the same real sequence (2025-11-12 storm-like interval) extended by 7 extra real frames to
cover both autoregressive steps. Colored borders preserve the observed / newly-predicted /
fed-back-as-input role coding.

In [1]:
import sys, os
sys.path.insert(0, '/users/framunno/projects/ionosphere_diffusion')

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch
from scipy.ndimage import map_coordinates

from src.data.dataset import IonoSequenceDataset, latlon_to_cartesian_grid

plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['font.family'] = 'STIXGeneral'

PAPER_FIGURES_DIR = '/users/framunno/projects/paper_writing/SuperDARN_deep_Learning_Francesco/figures'
CSV_PATH = '/users/framunno/data/ionosphere/l1_to_map_matched_2020_2025.csv'
MAPS_BASE = '/capstor/scratch/cscs/framunno/ionosphere_data/all_maps/'

N_COND, M_PRED, K_STEPS = 15, 7, 50
BLUE = '#3a6ea5'
ORANGE = '#c1662d'
ORANGE_FADED = '#e8c2a8'
GRAY = '0.55'


In [2]:
# Same real sequence as the training-strategy figure: pick the most active real, complete
# sequence in the validation split (locally available files only), same selection method.
SEQ_LEN = N_COND + M_PRED  # 22, matches the dataset's own construction
ds = IonoSequenceDataset(
    csv_path=CSV_PATH, transform_cond_csv=None, sequence_length=SEQ_LEN,
    normalization_type='absolute_max', use_l1_conditions=True, min_center_distance=15,
    cartesian_transform=True, output_size=128, only_complete_sequences=True, split='valid',
)
print('num complete valid sequences:', len(ds))

scores = []
TOTAL_FRAMES = N_COND + M_PRED + M_PRED  # 29, checked up front so the chosen sequence has all frames available
for i, center_idx in enumerate(ds.sequences):
    start_idx = center_idx - ds.sequence_length // 2
    bzs, speeds, ok = [], [], True
    for off in range(TOTAL_FRAMES):
        fidx = start_idx + off
        if fidx < 0 or fidx >= len(ds.all_files):
            ok = False; break
        fname = os.path.basename(ds.all_files[fidx])
        if fname not in ds.filename_to_conditions or not os.path.exists(MAPS_BASE + fname):
            ok = False; break
        bx, by, bz, vx = ds.filename_to_conditions[fname]
        bzs.append(bz); speeds.append(abs(vx))
    if not ok:
        continue
    scores.append((-np.mean(bzs) * np.mean(speeds), i, center_idx))
scores.sort(reverse=True)
chosen_seq_i = scores[0][1]
center_idx = ds.sequences[chosen_seq_i]
start_idx = center_idx - ds.sequence_length // 2
print('chosen center time:', ds.all_timestamps[center_idx])


Building L1 conditions cache for valid split...


Cached L1 conditions for 1370626 files


Filtered sequences for valid: 25299 -> 18312 (72.4% complete sequences)
num complete valid sequences: 18312


chosen center time: 2025-04-16 14:46:00


In [3]:
# Load N_COND+M_PRED+M_PRED = 29 consecutive REAL frames directly (extends 7 frames past the
# training-strategy window, to cover both AR steps with genuine observed data throughout).
NORM_SCALE = 80000
TOTAL_FRAMES = N_COND + M_PRED + M_PRED  # 29

frames_phys = []
ok_all = True
for off in range(TOTAL_FRAMES):
    fidx = start_idx + off
    if fidx < 0 or fidx >= len(ds.all_files):
        ok_all = False; break
    fpath = MAPS_BASE + os.path.basename(ds.all_files[fidx])
    if not os.path.exists(fpath):
        ok_all = False; break
    raw = np.load(fpath, allow_pickle=True)[0].astype(np.float32)
    cart = latlon_to_cartesian_grid(raw, output_size=128)
    frames_phys.append(np.clip(cart, -NORM_SCALE, NORM_SCALE))
print('loaded', len(frames_phys), 'frames, ok_all =', ok_all)
frames_phys = np.stack(frames_phys, axis=0)
VMAX = np.abs(frames_phys[:N_COND]).max()
print('VMAX:', VMAX)


loaded 29 frames, ok_all = True
VMAX: 35195.05681249158


In [4]:
H_IMG, MAX_R = 128, 24
r_i, th_i = np.linspace(0, MAX_R, 200), np.linspace(0, 2*np.pi, 360)
r_grid, theta_grid = np.meshgrid(r_i, th_i)
polar_x, polar_y = r_grid*np.cos(theta_grid), r_grid*np.sin(theta_grid)
col_coords = (polar_x + MAX_R) / (2*MAX_R) * (H_IMG - 1)
row_coords = (polar_y + MAX_R) / (2*MAX_R) * (H_IMG - 1)

def to_polar(frame2d):
    return map_coordinates(frame2d, [row_coords, col_coords], order=1, mode='constant', cval=0)

def draw_frame_row(fig, ax, x0, y0, frames, r=0.35, gap=0.9):
    """Places small real polar-plot insets (instead of flat circles) along a row, using the
    parent ax's data-coordinate system for layout so the row keeps the exact same geometry as
    the original schematic. border_colors codes each frame's role (observed / predicted / fed-back)."""
    x = x0
    inv = fig.transFigure.inverted()
    for frame in frames:
        cx_fig, cy_fig = inv.transform(ax.transData.transform((x, y0)))
        ex_fig, _ = inv.transform(ax.transData.transform((x + r, y0)))
        _, ey_fig = inv.transform(ax.transData.transform((x, y0 + r)))
        rx_fig, ry_fig = ex_fig - cx_fig, ey_fig - cy_fig
        axp = fig.add_axes([cx_fig - rx_fig, cy_fig - ry_fig, 2*rx_fig, 2*ry_fig], projection='polar')
        axp.set_ylim(0, MAX_R)
        axp.pcolormesh(theta_grid, r_grid, to_polar(frame), shading='auto', cmap='coolwarm',
                        vmin=-VMAX, vmax=VMAX, zorder=1)
        axp.set_theta_zero_location('S'); axp.set_theta_direction(1)
        axp.set_xticks([]); axp.set_yticks([]); axp.grid(False)
        axp.spines['polar'].set_linewidth(1.0)
        axp.spines['polar'].set_color('0.55')
        axp.patch.set_alpha(0)
        x += gap
    return x

def draw_zone_rect(ax, x0, n, gap, color, r_pad=0.5, y0=1.0, h_pad=0.55):
    """Light background rectangle spanning n frame slots starting at x0, drawn BEHIND the row."""
    from matplotlib.patches import Rectangle
    x_left = x0 - r_pad
    x_right = x0 + (n - 1) * gap + r_pad
    ax.add_patch(Rectangle((x_left, y0 - h_pad), x_right - x_left, 2 * h_pad,
                            facecolor=color, edgecolor='none', zorder=0))
    return x_right


In [5]:
from matplotlib.patches import Rectangle

LIGHT_GRAY = '#e8e8e8'
LIGHT_ORANGE = '#f7ddc4'
INPUT_N, OUTPUT_N = 4, 3          # simplified illustrative window (paper uses N=15, M=7)
WINDOW = INPUT_N + OUTPUT_N        # 7
SHIFT = OUTPUT_N                   # slides forward by the output size each step
gap = 0.9

all_frames_22 = [frames_phys[i] for i in range(N_COND + M_PRED)]  # the real 22-frame trajectory
starts = list(range(0, len(all_frames_22) - WINDOW + 1, SHIFT))   # 0, 3, 6, ..., 15 -> 6 rows
n_rows = len(starts)

fig, (axA, axB) = plt.subplots(2, 1, figsize=(15, 9.2), height_ratios=[0.55, 3.0],
                                gridspec_kw={'hspace': 0.1})

# ============================== Panel A: single-pass (only the window is shown) ==============
x_max = 1.0 + (len(all_frames_22) - 1) * gap + 1.0  # same scale as the autoregressive panel below
axA.set_xlim(0, x_max); axA.set_ylim(0, 2.0); axA.axis('off')
title_y = 1.7
axA.text(0.2, title_y, 'SINGLE-PASS', fontsize=17, fontweight='bold', ha='left', va='center',
         color='0.15', fontfamily='STIXGeneral')

# legend, on the same line as the title
leg_fs = 15
leg_x = x_max - 4.7
axA.add_patch(Rectangle((leg_x, title_y - 0.14), 0.42, 0.28, facecolor=LIGHT_GRAY, edgecolor='none'))
axA.text(leg_x + 0.55, title_y, 'Input', fontsize=leg_fs, ha='left', va='center', color='0.35', fontfamily='STIXGeneral')
axA.add_patch(Rectangle((leg_x + 1.9, title_y - 0.14), 0.42, 0.28, facecolor=LIGHT_ORANGE,
                         edgecolor='none'))
axA.text(leg_x + 2.45, title_y, 'Prediction', fontsize=leg_fs, ha='left', va='center', color='0.35', fontfamily='STIXGeneral')

draw_zone_rect(axA, 1.0, INPUT_N, gap, LIGHT_GRAY, y0=0.85)
draw_zone_rect(axA, 1.0 + INPUT_N * gap, OUTPUT_N, gap, LIGHT_ORANGE, y0=0.85)
draw_frame_row(fig, axA, 1.0, 0.85, all_frames_22[:WINDOW])

# ============================== Panel B: autoregressive, full trajectory, sliding window =======
row_gap_y = 1.0
axB.set_xlim(0, x_max); axB.set_ylim(0, n_rows * row_gap_y + 1.3); axB.axis('off')
axB.text(0.2, n_rows * row_gap_y + 1.0, 'AUTOREGRESSIVE ($K$ STEPS)', fontsize=17,
         fontweight='bold', ha='left', va='center', color='0.15', fontfamily='STIXGeneral')

for r, start in enumerate(starts):
    y = (n_rows - 1 - r) * row_gap_y + 1.0
    draw_zone_rect(axB, 1.0 + start * gap, INPUT_N, gap, LIGHT_GRAY, y0=y)
    draw_zone_rect(axB, 1.0 + (start + INPUT_N) * gap, OUTPUT_N, gap, LIGHT_ORANGE, y0=y)
    draw_frame_row(fig, axB, 1.0, y, all_frames_22)

plt.savefig(f'{PAPER_FIGURES_DIR}/methodology_ar_schematic.png', dpi=300, bbox_inches='tight')
plt.show()


/tmp/ipykernel_20123/236833453.py:24: MatplotlibDeprecationWarning: Auto-removal of grids by pcolor() and pcolormesh() is deprecated since 3.5 and will be removed two minor releases later; please call grid(False) first.
  axp.pcolormesh(theta_grid, r_grid, to_polar(frame), shading='auto', cmap='coolwarm',
